In [ ]:
import csv
import os
import struct
import time
from pathlib import Path
import numpy as np
import scipy.sparse as sp
from pynq import MMIO, Overlay, allocate

# ---------------------------------------------------------------------
# Configuration & Constants
# ---------------------------------------------------------------------
PACK_SIZE = 16
NUM_ROWS = 32768
NUM_COLS = 32768

ADMM_IP_CONTROL_BASE = 0xA0000000
ADMM_IP_CONTROL_R_BASE = 0xA0010000
ADDR_RANGE = 0x20000

DATA_ROOT = "/home/xilinx/jupyter_notebooks/dse/32768x32768_data_bin_dse"
BITSTREAM_ROOT = "/home/xilinx/jupyter_notebooks/dse/generated_bitstreams"
OUTPUT_CSV = "/home/xilinx/jupyter_notebooks/dse/exp3/exp3_tile_size_sweep_results.csv"

DENSITIES = ["low", "medium", "high"]
ADAPTIVE_RHO_VALUES = [0]
NUM_SOLVES_PER_SETTING = 1

BITSTREAM_CONFIGS = [
    # --- 1024x1024 Configurations ---
    {"label": "1024x1024_reshape1_pes16", "tile": 1024, "pes": 16},
    {"label": "1024x1024_reshape1_pes20", "tile": 1024, "pes": 20},
    {"label": "1024x1024_reshape1_pes30", "tile": 1024, "pes": 30},
    {"label": "1024x1024_reshape8_pes16", "tile": 1024, "pes": 16},
    {"label": "1024x1024_reshape8_pes20", "tile": 1024, "pes": 20},
    {"label": "1024x1024_reshape8_pes30", "tile": 1024, "pes": 30},

    # --- 4096x4096 Configurations ---
    {"label": "4096x4096_reshape1_pes16", "tile": 4096, "pes": 16},
    {"label": "4096x4096_reshape1_pes20", "tile": 4096, "pes": 20},
    {"label": "4096x4096_reshape1_pes30", "tile": 4096, "pes": 30},
    {"label": "4096x4096_reshape8_pes16", "tile": 4096, "pes": 16},
    {"label": "4096x4096_reshape8_pes20", "tile": 4096, "pes": 20},
    {"label": "4096x4096_reshape8_pes30", "tile": 4096, "pes": 30},

    # --- 8192x8192 Configurations ---
    {"label": "8192x8192_reshape1_pes16", "tile": 8192, "pes": 16},
    {"label": "8192x8192_reshape1_pes20", "tile": 8192, "pes": 20},
    {"label": "8192x8192_reshape1_pes30", "tile": 8192, "pes": 30},
    {"label": "8192x8192_reshape8_pes16", "tile": 8192, "pes": 16},
    {"label": "8192x8192_reshape8_pes20", "tile": 8192, "pes": 20},
    {"label": "8192x8192_reshape8_pes30", "tile": 8192, "pes": 30},

    # --- 16384x16384 Configurations ---
    {"label": "16384x16384_reshape1_pes12", "tile": 16384, "pes": 12},
    {"label": "16384x16384_reshape8_pes12", "tile": 16384, "pes": 12},
    {"label": "16384x16384_reshape8_pes16", "tile": 16384, "pes": 16},
    {"label": "16384x16384_reshape8_pes20", "tile": 16384, "pes": 20},

    # --- 32768x32768 Configurations ---
    {"label": "32768x32768_reshape1_pes6", "tile": 32768, "pes": 6},
    {"label": "32768x32768_reshape8_pes6", "tile": 32768, "pes": 6},
    {"label": "32768x32768_reshape8_pes8", "tile": 32768, "pes": 8},
    {"label": "32768x32768_reshape8_pes12", "tile": 32768, "pes": 12},
]

alpha = 1.8
sigma = 1e-2
admm_max_iterations = 2000
pcg_max_iterations = 5
eps_abs = 1e-3
eps_rel = 1e-3
pcg_tol_frac = 1.0

# ---------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------
def float_to_uint(f_val): return struct.unpack("<I", struct.pack("<f", float(f_val)))[0]
def uint_to_float(u_val): return struct.unpack("<f", struct.pack("<I", int(u_val) & 0xFFFFFFFF))[0]

def write_64bit_address(ip, base_offset, address):
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

def ceil_div(a, b): return (a + b - 1) // b

def flush_denormals(arr):
    mask = (np.abs(arr) > 0.0) & (np.abs(arr) < 1e-15)
    arr[mask] = 0.0
    return arr

def allocate_and_copy(np_array, dtype):
    buf = allocate(shape=np_array.shape, dtype=dtype, cacheable=False)
    buf[:] = np_array[:]
    return buf

def apply_scaling(P_d, A_sp, q_vec, l_vec, u_vec, iters=10):
    n, m = len(P_d), A_sp.shape[0]
    E, D = np.ones(n, dtype=np.float32), np.ones(m, dtype=np.float32)
    P_scaled, A_scaled = P_d.copy(), A_sp.copy()
    for _ in range(iters):
        A_abs = abs(A_scaled)
        A_col_norm = A_abs.max(axis=0).toarray().flatten()
        x_norm = np.maximum(np.maximum(np.abs(P_scaled), A_col_norm), 1e-4)
        z_norm = np.maximum(A_abs.max(axis=1).toarray().flatten(), 1e-4)
        E_new, D_new = 1.0 / np.sqrt(x_norm), 1.0 / np.sqrt(z_norm)
        E, D = E * E_new, D * D_new
        P_scaled = P_scaled * (E_new ** 2)
        A_scaled = sp.diags(D_new).dot(A_scaled).dot(sp.diags(E_new)).tocsc()
    q_scaled = q_vec * E
    l_scaled = np.where(l_vec <= -1e17 * 0.9, -1e17, l_vec * D)
    u_scaled = np.where(u_vec >= 1e17 * 0.9, 1e17, u_vec * D)
    c = max(1.0 / np.maximum(np.max(np.abs(P_scaled)), np.max(np.abs(q_scaled))), 1e-4)
    return P_scaled * c, A_scaled, q_scaled * c, l_scaled, u_scaled, E, D, c

def build_tiled_csc(sp_mat, tile_rows, tile_cols, pack_size=16):
    global_rows, global_cols = sp_mat.shape
    num_row_tiles = ceil_div(global_rows, tile_rows)
    num_col_tiles = ceil_div(global_cols, tile_cols)
    num_tiles = num_row_tiles * num_col_tiles
    
    g_col_ptr, g_row_idx, g_values = sp_mat.indptr, sp_mat.indices, sp_mat.data
    
    # 1. Distribute elements into tile bins
    tiles_rows = [[[] for _ in range(tile_cols)] for _ in range(num_tiles)]
    tiles_vals = [[[] for _ in range(tile_cols)] for _ in range(num_tiles)]
    
    for c in range(global_cols):
        tc, local_c = c // tile_cols, c % tile_cols
        for idx in range(g_col_ptr[c], g_col_ptr[c + 1]):
            r, v = g_row_idx[idx], g_values[idx]
            tr, local_r = r // tile_rows, r % tile_rows
            tile_idx = tr * num_col_tiles + tc
            tiles_rows[tile_idx][local_c].append(local_r)
            tiles_vals[tile_idx][local_c].append(v)
            
    t_cnt, t_noff, t_coff, c_ptr = [], [], [], []
    r_idx_all, vals_all = [], []
    nnz_cursor = 0
    
    # 2. Vectorized block packing
    for t_idx in range(num_tiles):
        t_coff.append(len(c_ptr))
        local_ptr = [0] * (tile_cols + 1)
        
        tr, tv = [], []
        t_nnz = 0
        for c in range(tile_cols):
            tr.extend(tiles_rows[t_idx][c])
            tv.extend(tiles_vals[t_idx][c])
            t_nnz += len(tiles_rows[t_idx][c])
            local_ptr[c + 1] = t_nnz
            
        t_cnt.append(t_nnz)
        t_noff.append(nnz_cursor)
        c_ptr.extend(local_ptr)
        
        words = ceil_div(t_nnz, pack_size)
        if words > 0:
            pad_len = words * pack_size - t_nnz
            
            # Convert directly to numpy arrays
            tr_arr = np.array(tr, dtype=np.int32)
            tv_arr = np.array(tv, dtype=np.float32)
            
            # Pad the ends with zeros
            if pad_len > 0:
                tr_arr = np.pad(tr_arr, (0, pad_len), mode='constant', constant_values=0)
                tv_arr = np.pad(tv_arr, (0, pad_len), mode='constant', constant_values=0.0)
            
            # Reshape into (words, pack_size)
            r_idx_all.append(tr_arr.reshape(words, pack_size))
            vals_all.append(tv_arr.reshape(words, pack_size))
            
        nnz_cursor += words

    # Flatten all packed matrices
    if len(r_idx_all) > 0:
        final_r_idx = np.concatenate(r_idx_all)
        final_vals = np.concatenate(vals_all)
    else:
        final_r_idx = np.empty((0, pack_size), dtype=np.int32)
        final_vals = np.empty((0, pack_size), dtype=np.float32)

    return (num_row_tiles, num_col_tiles, 
            np.array(t_cnt, dtype=np.int32), 
            np.array(t_noff, dtype=np.int32), 
            np.array(t_coff, dtype=np.int32), 
            np.array(c_ptr, dtype=np.int32), 
            final_r_idx, final_vals)

def load_problem(density_label):
    density_map = {"low": "low", "medium": "med", "high": "high"}
    case_dir = os.path.join(DATA_ROOT, density_map[density_label])
    def load_bin(n, dt): return np.fromfile(os.path.join(case_dir, n), dtype=dt)
    A_s = sp.csc_matrix((load_bin('A_data.bin', np.float32), load_bin('A_indices.bin', np.int32), load_bin('A_indptr.bin', np.int32)), shape=(NUM_ROWS, NUM_COLS))
    P_s = sp.csc_matrix((load_bin('P_data.bin', np.float32), load_bin('P_indices.bin', np.int32), load_bin('P_indptr.bin', np.int32)), shape=(NUM_COLS, NUM_COLS))
    q, l, u = load_bin('q.bin', np.float32)[:NUM_COLS], load_bin('l.bin', np.float32)[:NUM_ROWS], load_bin('u.bin', np.float32)[:NUM_ROWS]
    x_true = load_bin('x_true.bin', np.float32)[:NUM_COLS]
    P_diag = load_bin('P_diag.bin', np.float32)[:NUM_COLS] if os.path.exists(os.path.join(case_dir, 'P_diag.bin')) else P_s.diagonal().astype(np.float32)
    Pd_s, As_s, qs_s, ls_s, us_s, E, D, c = apply_scaling(P_diag, A_s, q, l, u)
    As_s.data, qs_s, Pd_s = flush_denormals(As_s.data), flush_denormals(qs_s), flush_denormals(Pd_s)
    return {"A": As_s, "AT": As_s.T.tocsc(), "P": sp.diags(Pd_s).tocsc(), "Pd": Pd_s, "q": qs_s, 
            "l": ls_s, "u": us_s, "x_t": x_true, "E": E, "nnz": As_s.nnz}, NUM_ROWS, NUM_COLS

# ---------------------------------------------------------------------
# Main Sweep Execution
# ---------------------------------------------------------------------
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

with open(OUTPUT_CSV, mode="w", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["density", "config", "tile_size", "pes", "adaptive_rho", "solve_index", "solve_ms", "status", "mae", "admm_iterations", "pcg_iterations", "r_prim", "r_dual"])
    writer.writeheader()
    
    for d_label in DENSITIES:
        print(f"\n>>> Density: {d_label}")
        prob, nr, nc = load_problem(d_label)

        last_tile_dim = None 
        tiled_mats = None
        
        for cfg in BITSTREAM_CONFIGS:
            tile_dim = cfg["tile"]

            if tile_dim != last_tile_dim:
                print(f"--- Slicing Tiled Matrices for NEW Tile Size: {tile_dim}x{tile_dim} ---")
                tiled_mats = {n: build_tiled_csc(prob[n], tile_dim, tile_dim, PACK_SIZE) for n in ["A", "AT", "P"]}
                last_tile_dim = tile_dim
            else:
                print(f"--- Reusing Tiled Matrices for existing Tile Size: {tile_dim}x{tile_dim} ---")
            
            bit_path = os.path.join(BITSTREAM_ROOT, cfg["label"], f"admm_{cfg['label']}.bit")
            print(f"--- Loading Bitstream: {cfg['label']} ---")
            
            if 'overlay' in locals():
                del overlay  # Explicitly clear the previous overlay to free mapped resources
                
            overlay = Overlay(bit_path)
            ctrl, rctrl = MMIO(ADMM_IP_CONTROL_BASE, ADDR_RANGE), MMIO(ADMM_IP_CONTROL_R_BASE, ADDR_RANGE)
            hw_bufs = []
            
            def alloc_hw(arr, dt):
                b = allocate_and_copy(arr, dt); hw_bufs.append(b); return b
            
            # 1. Contiguous A Buffers
            A_row_words = np.zeros((ceil_div(prob["nnz"], PACK_SIZE), PACK_SIZE), dtype=np.int32)
            A_val_words = np.zeros((ceil_div(prob["nnz"], PACK_SIZE), PACK_SIZE), dtype=np.float32)
            A_row_words.flat[:prob["nnz"]] = prob["A"].indices
            A_val_words.flat[:prob["nnz"]] = prob["A"].data
            A_r_hw, A_v_hw, A_c_hw = alloc_hw(A_row_words, np.int32), alloc_hw(A_val_words, np.float32), alloc_hw(prob["A"].indptr, np.int32)
            
            # 2. Tiling Buffer Mapping
            tm = {}
            for n in ["A", "AT", "P"]:
                tm[n] = [alloc_hw(tiled_mats[n][i+2], [np.int32, np.int32, np.int32, np.int32, np.int32, np.float32][i]) for i in range(6)]
            
            # 3. Vector Buffers
            rho_in = np.full(nr, 1.0, dtype=np.float32)
            rho_in[~((np.abs(prob["l"]) < 9e16) & (np.abs(prob["u"]) < 9e16))] = 1e-6
            Pd_hw, l_hw, u_hw, q_hw = alloc_hw(prob["Pd"], np.float32), alloc_hw(prob["l"], np.float32), alloc_hw(prob["u"], np.float32), alloc_hw(prob["q"], np.float32)
            rho_hw = alloc_hw(rho_in, np.float32)
            x_hw, y_hw = allocate((nc,), np.float32), allocate((nr,), np.float32)
            hw_bufs.extend([x_hw, y_hw])
            
            for adp in ADAPTIVE_RHO_VALUES:
                print(f"      adaptive_rho={adp}")
                # Configuration Registers
                ctrl.write(0x10, nr); ctrl.write(0x18, nc); ctrl.write(0x20, prob["nnz"])
                
                # Write actual tile counts
                ctrl.write(0x28, tiled_mats["A"][0]); ctrl.write(0x30, tiled_mats["A"][1])
                ctrl.write(0x38, tiled_mats["AT"][0]); ctrl.write(0x40, tiled_mats["AT"][1])
                ctrl.write(0x48, tiled_mats["P"][0]); ctrl.write(0x50, tiled_mats["P"][1])
                
                ctrl.write(0x58, float_to_uint(sigma)); ctrl.write(0x60, float_to_uint(alpha))
                ctrl.write(0x68, admm_max_iterations); ctrl.write(0x70, pcg_max_iterations); ctrl.write(0x78, adp)
                ctrl.write(0x80, float_to_uint(eps_abs)); ctrl.write(0x88, float_to_uint(eps_rel)); ctrl.write(0x90, float_to_uint(pcg_tol_frac))
                
                # --- START ADDRESS MAPPING ---
                write_64bit_address(rctrl, 0x010, A_r_hw.device_address)
                write_64bit_address(rctrl, 0x01c, A_c_hw.device_address)
                write_64bit_address(rctrl, 0x028, A_v_hw.device_address)
                
                # A Tiled Matrix
                write_64bit_address(rctrl, 0x034, tm["A"][0].device_address) 
                write_64bit_address(rctrl, 0x040, tm["A"][1].device_address) 
                write_64bit_address(rctrl, 0x04c, tm["A"][2].device_address) 
                write_64bit_address(rctrl, 0x058, tm["A"][4].device_address) 
                write_64bit_address(rctrl, 0x064, tm["A"][3].device_address) 
                write_64bit_address(rctrl, 0x070, tm["A"][5].device_address) 
                
                # AT Tiled Matrix
                write_64bit_address(rctrl, 0x07c, tm["AT"][0].device_address)
                write_64bit_address(rctrl, 0x088, tm["AT"][1].device_address)
                write_64bit_address(rctrl, 0x094, tm["AT"][2].device_address)
                write_64bit_address(rctrl, 0x0a0, tm["AT"][4].device_address)
                write_64bit_address(rctrl, 0x0ac, tm["AT"][3].device_address)
                write_64bit_address(rctrl, 0x0b8, tm["AT"][5].device_address)
                
                # P Tiled Matrix
                write_64bit_address(rctrl, 0x0c4, tm["P"][0].device_address)
                write_64bit_address(rctrl, 0x0d0, tm["P"][1].device_address)
                write_64bit_address(rctrl, 0x0dc, tm["P"][2].device_address)
                write_64bit_address(rctrl, 0x0e8, tm["P"][4].device_address)
                write_64bit_address(rctrl, 0x0f4, tm["P"][3].device_address)
                write_64bit_address(rctrl, 0x100, tm["P"][5].device_address)
                
                # Vectors
                write_64bit_address(rctrl, 0x10c, Pd_hw.device_address); write_64bit_address(rctrl, 0x118, l_hw.device_address)
                write_64bit_address(rctrl, 0x124, u_hw.device_address); write_64bit_address(rctrl, 0x130, q_hw.device_address)
                write_64bit_address(rctrl, 0x13c, rho_hw.device_address); write_64bit_address(rctrl, 0x148, x_hw.device_address)
                write_64bit_address(rctrl, 0x154, y_hw.device_address)
                # --- END ADDRESS MAPPING ---
                
                for i in range(NUM_SOLVES_PER_SETTING):
                    x_hw[:], y_hw[:] = 0.0, 0.0
                    rho_hw[:] = rho_in[:]
                    ts = time.perf_counter()
                    ctrl.write(0x00, 0x01)
                    while (ctrl.read(0x00) & 0x02) == 0: pass
                    ms = (time.perf_counter() - ts) * 1000.0
                    
                    admm_iters = int(ctrl.read(0x98))
                    pcg_iters = int(ctrl.read(0xa8))
                    r_prim = uint_to_float(ctrl.read(0xb8))
                    r_dual = uint_to_float(ctrl.read(0xc8))
                    status = int(ctrl.read(0xd8))
                    
                    mae = np.mean(np.abs((np.array(x_hw) * prob["E"]) - prob["x_t"]))
                    writer.writerow({
                        "density": d_label, 
                        "config": cfg["label"], 
                        "tile_size": cfg["tile"], 
                        "pes": cfg["pes"], 
                        "adaptive_rho": adp, 
                        "solve_index": i+1, 
                        "solve_ms": ms, 
                        "status": status, 
                        "mae": mae, 
                        "admm_iterations": admm_iters, 
                        "pcg_iterations": pcg_iters, 
                        "r_prim": r_prim, 
                        "r_dual": r_dual
                    })
                    csv_file.flush()
            
            # Clean up buffers at the end of EACH config
            for b in hw_bufs: 
                b.freebuffer()

print(f"\nExperiment 3 Sweep complete. Data saved to {OUTPUT_CSV}")